# Optimizer Benchmarking & Model Compression for Edge AI
**Option 2 - Excellent Quality** | 6-Stage Pipeline | RTX 6000 Ada (48 GB)

| Stage | Description |
|-------|-------------|
| 1 | Model & Dataset selection with justification |
| 2 | Fine-tuning with 5 optimizers + LR sensitivity + gradient norms + energy tracking |
| 3 | Comprehensive evaluation before compression |
| 4 | Compression: INT8 quantization + unstructured pruning + ONNX export |
| 5 | Evaluation after compression |
| 6 | Comparative table, Pareto front, recommendation matrix |

## Setup & Dependencies

In [ ]:
!pip install -q transformers datasets accelerate codecarbon onnx onnxruntime matplotlib seaborn pandas tqdm lion-pytorch psutil

### Imports & Configuration

In [ ]:
import os, json, time, math, copy, warnings, gc, pickle
from collections import defaultdict
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm, trange
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW, SGD
from transformers import (AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling,
                          get_scheduler, set_seed, Adafactor)
from datasets import load_dataset
from codecarbon import EmissionsTracker
from lion_pytorch import Lion

warnings.filterwarnings('ignore')
set_seed(42)
sns.set_style('whitegrid')

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1e9))


In [ ]:
# ========== Configuration — optimised for RTX 6000 Ada (48 GB) ==========
CONFIG = {
    'model_name': 'distilgpt2',
    'dataset_name': 'roneneldan/TinyStories',
    'max_length': 256,
    'batch_size': 128,
    'batch_size_eval': 256,
    'num_epochs': 3,
    'learning_rates': [1e-4, 3e-4, 1e-3],
    'default_lr': 3e-4,
    'max_train_samples': 100000,
    'max_val_samples': 10000,
    'warmup_steps': 200,
    'gradient_accumulation_steps': 1,
    'num_workers': 4,
    'output_dir': './results',
}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print("Config:", json.dumps(CONFIG, indent=2))
print(f"Device: {device}")


---
## Stage 1: Model & Dataset Selection

### Model: DistilGPT-2 (82M parameters)
- Lightweight distilled variant of GPT-2. Realistic for edge deployment.

### Dataset: TinyStories
- 2.7M short synthetic stories. Subset: 100K train / 10K val.


In [ ]:
# ---- Load Model & Tokenizer ----
print(f"Loading model: {CONFIG['model_name']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG['model_name']).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Model size (FP32): {total_params * 4 / 1e6:.1f} MB")


In [ ]:
# ---- Load Dataset ----
print(f"Loading dataset: {CONFIG['dataset_name']}")
dataset = load_dataset(CONFIG['dataset_name'], split='train')
print(f"Total examples: {len(dataset):,}")

def tokenize_fn(examples):
    texts = examples['text']   # batched=True -> dict of lists
    enc = tokenizer(texts, truncation=True, padding='max_length',
                    max_length=CONFIG['max_length'], return_tensors=None)
    enc['labels'] = enc['input_ids'].copy()
    return enc

dataset = dataset.select(range(CONFIG['max_train_samples'] + CONFIG['max_val_samples']))
dataset = dataset.map(tokenize_fn, batched=True, remove_columns=['text'], batch_size=1000)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

train_dataset = dataset.select(range(CONFIG['max_train_samples']))
val_dataset   = dataset.select(range(CONFIG['max_train_samples'],
                                     CONFIG['max_train_samples'] + CONFIG['max_val_samples']))
print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")

collator    = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True,  collate_fn=collator,
                          num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size_eval'],
                          shuffle=False, collate_fn=collator,
                          num_workers=CONFIG['num_workers'], pin_memory=True)
print("Data loaders ready.")


---
## Stage 2: Fine-tuning with 5 Optimizers

| Optimizer | Key Characteristic | Memory / param |
|-----------|-------------------|---------------|
| **AdamW** | Adam + decoupled weight decay | 8 bytes |
| **Adafactor** | Factorised second moments | ~3 bytes |
| **Lion** | Sign-based update | 4 bytes |
| **LAMB** | Layer-wise adaptive rates | 8 bytes |
| **SGD** | Vanilla + momentum | 4 bytes |


In [ ]:
# ---- Optimizer Factory ----
def get_optimizer(name, model, lr):
    if name == 'AdamW':
        return AdamW(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)
    elif name == 'Adafactor':
        return Adafactor(model.parameters(), lr=lr, scale_parameter=True,
                         relative_step=False, warmup_init=False, weight_decay=0.01)
    elif name == 'Lion':
        return Lion(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.01)
    elif name == 'LAMB':
        from torch.optim import Optimizer

        class LAMB(Optimizer):
            def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
                defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
                super().__init__(params, defaults)

            def step(self, closure=None):
                loss = None
                if closure is not None:
                    loss = closure()
                for group in self.param_groups:
                    for p in group['params']:
                        if p.grad is None:
                            continue
                        grad = p.grad.data
                        state = self.state[p]
                        if len(state) == 0:
                            state['step'] = 0
                            state['exp_avg']    = torch.zeros_like(p.data)
                            state['exp_avg_sq'] = torch.zeros_like(p.data)
                        ea, esq = state['exp_avg'], state['exp_avg_sq']
                        b1, b2  = group['betas']
                        state['step'] += 1
                        ea.mul_(b1).add_(grad, alpha=1 - b1)
                        esq.mul_(b2).addcmul_(grad, grad, value=1 - b2)
                        bc1 = 1 - b1 ** state['step']
                        bc2 = 1 - b2 ** state['step']
                        step = ea / (esq.sqrt() + group['eps'])
                        step /= bc1 / (bc2 ** 0.5)   # FIX: bc2 is a float, use ** 0.5
                        if group['weight_decay'] > 0:
                            step.add_(p.data, alpha=group['weight_decay'])
                        r1 = torch.norm(p.data).clamp(min=1e-8)
                        r2 = torch.norm(step).clamp(min=1e-8)
                        p.data.add_(step, alpha=-group['lr'] * r1 / r2)
                return loss

        return LAMB(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)
    elif name == 'SGD':
        return SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.01)
    raise ValueError(f"Unknown optimizer: {name}")

OPTIMIZER_NAMES = ['AdamW', 'Adafactor', 'Lion', 'LAMB', 'SGD']
print("Optimizer factory ready. Available:", OPTIMIZER_NAMES)


In [ ]:
# ---- Training Function with AMP ----
_scaler = torch.cuda.amp.GradScaler()

def train_epoch(model, loader, optimizer, scheduler, device, gradient_accumulation_steps=1,
                track_grad_norms=False, track_energy=False):
    model.train()
    total_loss = 0.0; all_grad_norms = []; step_times = []; num_steps = 0
    use_amp = device.type == 'cuda'
    tracker = None
    if track_energy:
        tracker = EmissionsTracker(project_name=f"distilgpt2_{optimizer.__class__.__name__}",
                                   measure_power_secs=5, output_dir=CONFIG['output_dir'],
                                   save_to_file=True, log_level='error')
        tracker.start()
    pbar = tqdm(loader, desc=f"Training {optimizer.__class__.__name__}", leave=False)
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(pbar):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        t0 = time.perf_counter()
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss = model(**batch).loss / gradient_accumulation_steps
        _scaler.scale(loss).backward()
        torch.cuda.synchronize()
        step_times.append(time.perf_counter() - t0)
        if track_grad_norms:
            _scaler.unscale_(optimizer)
            total_norm = sum(p.grad.data.norm(2).item() ** 2
                             for p in model.parameters() if p.grad is not None)
            all_grad_norms.append(total_norm ** 0.5)
        if (step + 1) % gradient_accumulation_steps == 0:
            if not track_grad_norms:
                _scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            _scaler.step(optimizer); _scaler.update()
            scheduler.step(); optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item() * gradient_accumulation_steps; num_steps += 1
        pbar.set_postfix({'loss': f'{total_loss / num_steps:.4f}'})
    if num_steps % gradient_accumulation_steps != 0:
        if not track_grad_norms:
            _scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        _scaler.step(optimizer); _scaler.update()
        scheduler.step(); optimizer.zero_grad(set_to_none=True)
    energy_kwh = tracker.stop() if track_energy and tracker else None
    avg_step_time = np.mean(step_times) if step_times else 0.0
    return {'avg_loss': total_loss / num_steps, 'grad_norms': all_grad_norms,
            'avg_step_time': avg_step_time, 'energy_kwh': energy_kwh, 'num_steps': num_steps}


In [ ]:
# ---- Perplexity Evaluation ----
@torch.no_grad()
def evaluate_perplexity(model, loader, device, max_batches=None):
    model.eval(); total_loss = 0.0; n = 0
    for batch in tqdm(loader, desc="Evaluating", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        total_loss += model(**batch).loss.item(); n += 1
        if max_batches and n >= max_batches:
            break
    avg_loss = total_loss / n
    return math.exp(avg_loss), avg_loss


### Run 1: Default LR Comparison (5 Optimizers)

In [ ]:
# ---- Run 1: All 5 Optimizers at Default LR ----
results_default = {}
lr_default = CONFIG['default_lr']

for opt_name in OPTIMIZER_NAMES:
    print(f"\n{'='*60}\nTraining with {opt_name} (lr={lr_default})\n{'='*60}")
    model_ = AutoModelForCausalLM.from_pretrained(CONFIG['model_name']).to(device)
    model_.train()
    optimizer = get_optimizer(opt_name, model_, lr_default)
    num_steps = len(train_loader) // CONFIG['gradient_accumulation_steps']
    scheduler = get_scheduler('linear', optimizer=optimizer,
                              num_warmup_steps=CONFIG['warmup_steps'],
                              num_training_steps=num_steps)
    metrics = train_epoch(model_, train_loader, optimizer, scheduler, device,
                          gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
                          track_grad_norms=True, track_energy=True)
    ppl, val_loss = evaluate_perplexity(model_, val_loader, device)
    metrics['val_perplexity'] = ppl; metrics['val_loss'] = val_loss
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        metrics['peak_memory_mb'] = torch.cuda.max_memory_allocated(device) / 1e6
    else:
        metrics['peak_memory_mb'] = 0
    results_default[opt_name] = metrics
    del model_, optimizer, scheduler; gc.collect(); torch.cuda.empty_cache()
    print(f"  {opt_name}: loss={metrics['avg_loss']:.4f}, ppl={ppl:.2f}, "
          f"energy={metrics['energy_kwh'] or 0:.4f} kWh, mem={metrics['peak_memory_mb']:.0f} MB")

# ---- Safety save after Stage 2 ----
with open(f"{CONFIG['output_dir']}/results_default.pkl", 'wb') as f:
    pickle.dump(results_default, f)
print("\nResults saved to results/results_default.pkl")
print("="*60 + "\nDefault LR comparison complete!\n" + "="*60)


### Training Curves: Default LR

In [ ]:
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
names  = list(results_default.keys())
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

ppls = [results_default[n]['val_perplexity'] for n in names]
bars = ax[0].bar(names, ppls, color=colors, edgecolor='black', linewidth=1.2)
for bar, ppl in zip(bars, ppls):
    ax[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
               f'{ppl:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax[0].set_title('Validation Perplexity (lower is better)', fontsize=13, fontweight='bold')
ax[0].set_ylabel('Perplexity'); ax[0].set_ylim(0, max(ppls) * 1.2)

energies = [results_default[n]['energy_kwh'] or 0 for n in names]
ax[1].bar(names, energies, color=colors, edgecolor='black', linewidth=1.2)
ax[1].set_title('Energy Consumption (lower is better)', fontsize=13, fontweight='bold')
ax[1].set_ylabel('kWh')
for i, e in enumerate(energies):
    if e > 0: ax[1].text(i, e + 0.001, f'{e:.4f}', ha='center', va='bottom', fontsize=9)

mems = [results_default[n]['peak_memory_mb'] for n in names]
ax[2].bar(names, mems, color=colors, edgecolor='black', linewidth=1.2)
ax[2].set_title('Peak GPU Memory (lower is better)', fontsize=13, fontweight='bold')
ax[2].set_ylabel('MB')
for i, m in enumerate(mems):
    ax[2].text(i, m + 10, f'{m:.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/optimizer_comparison_default_lr.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for opt_name in OPTIMIZER_NAMES:
    norms = results_default[opt_name]['grad_norms']
    if norms:
        smoothed = np.convolve(norms, np.ones(10) / 10, mode='valid')
        ax.plot(smoothed, label=opt_name, linewidth=1.5, alpha=0.85)
ax.set_title('Gradient L2 Norms During Training (smoothed)', fontsize=13, fontweight='bold')
ax.set_xlabel('Training Step'); ax.set_ylabel('Gradient Norm')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/gradient_norms.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")


### Run 2: Learning Rate Sensitivity Analysis (3 LR values per optimizer)

In [ ]:
lrs = CONFIG['learning_rates']
results_lr = {}
for opt_name in OPTIMIZER_NAMES:
    results_lr[opt_name] = {}
    for lr in lrs:
        print(f"  {opt_name} | lr={lr:.1e}")
        model_ = AutoModelForCausalLM.from_pretrained(CONFIG['model_name']).to(device)
        model_.train()
        optimizer = get_optimizer(opt_name, model_, lr)
        num_steps = len(train_loader) // CONFIG['gradient_accumulation_steps']
        scheduler = get_scheduler('linear', optimizer=optimizer,
                                  num_warmup_steps=CONFIG['warmup_steps'],
                                  num_training_steps=num_steps)
        metrics = train_epoch(model_, train_loader, optimizer, scheduler, device,
                              gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
                              track_grad_norms=False, track_energy=False)
        ppl, val_loss = evaluate_perplexity(model_, val_loader, device)
        metrics['val_perplexity'] = ppl; metrics['val_loss'] = val_loss
        results_lr[opt_name][lr] = metrics
        del model_, optimizer, scheduler; gc.collect(); torch.cuda.empty_cache()
        print(f"    ppl={ppl:.2f}, val_loss={val_loss:.4f}")

with open(f"{CONFIG['output_dir']}/results_lr.pkl", 'wb') as f:
    pickle.dump(results_lr, f)
print("\nLR sensitivity complete. Saved to results/results_lr.pkl")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
markers = ['o', 's', 'D', '^', 'v']
for i, opt_name in enumerate(OPTIMIZER_NAMES):
    lr_vals  = list(results_lr[opt_name].keys())
    ppl_vals = [results_lr[opt_name][lr]['val_perplexity'] for lr in lr_vals]
    ax.plot([str(lr) for lr in lr_vals], ppl_vals, marker=markers[i],
            label=opt_name, linewidth=2, markersize=8, color=colors[i])
    for lr, ppl in zip(lr_vals, ppl_vals):
        ax.annotate(f'{ppl:.1f}', (str(lr), ppl),
                    textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)
ax.set_title('LR Sensitivity: Perplexity vs Learning Rate', fontsize=13, fontweight='bold')
ax.set_xlabel('Learning Rate'); ax.set_ylabel('Validation Perplexity')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/lr_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")


---
## Stage 3: Comprehensive Evaluation (Before Compression)

Metrics: perplexity, p50/p95 latency, tokens/sec, GPU memory, CPU edge simulation, generated text.


In [ ]:
# ---- Select Best Model & Retrain ----
best_opt = min(results_default, key=lambda k: results_default[k]['val_perplexity'])
print(f"Best optimizer: {best_opt} | Val perplexity: {results_default[best_opt]['val_perplexity']:.2f}")

best_model = AutoModelForCausalLM.from_pretrained(CONFIG['model_name']).to(device)
best_optimizer = get_optimizer(best_opt, best_model, CONFIG['default_lr'])
num_steps = len(train_loader) // CONFIG['gradient_accumulation_steps']
scheduler = get_scheduler('linear', optimizer=best_optimizer,
                          num_warmup_steps=CONFIG['warmup_steps'],
                          num_training_steps=num_steps)
print("Retraining best model...")
best_metrics = train_epoch(best_model, train_loader, best_optimizer, scheduler, device,
                           gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
                           track_grad_norms=True, track_energy=True)
best_model.eval()

# Safety save — reload with: best_model.load_state_dict(torch.load('results/best_model_fp32.pt'))
torch.save(best_model.state_dict(), f"{CONFIG['output_dir']}/best_model_fp32.pt")
print("Best model saved to results/best_model_fp32.pt")

param_count  = sum(p.numel() for p in best_model.parameters())
size_fp32_mb = param_count * 4 / 1e6
print(f"Model size (FP32): {size_fp32_mb:.1f} MB")


In [ ]:
# ---- Detailed Inference Benchmark ----
@torch.no_grad()
def benchmark_inference(model, loader, device, num_warmup=3):
    model.eval(); latencies = []; total_tokens = 0
    for i, batch in enumerate(loader):
        if i >= num_warmup: break
        _ = model(**{k: v.to(device) for k, v in batch.items()})
        if device == 'cuda' or (hasattr(device, 'type') and device.type == 'cuda'):
            torch.cuda.synchronize()
    for batch in tqdm(loader, desc="Benchmarking"):
        batch = {k: v.to(device) for k, v in batch.items()}
        t0 = time.perf_counter()
        _ = model(**batch)
        if device == 'cuda' or (hasattr(device, 'type') and device.type == 'cuda'):
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - t0)
        total_tokens += batch['input_ids'].size(0) * batch['input_ids'].size(1)
    lat = np.array(latencies)
    return {'latency_p50_ms': np.percentile(lat, 50) * 1000,
            'latency_p95_ms': np.percentile(lat, 95) * 1000,
            'latency_mean_ms': lat.mean() * 1000,
            'tokens_per_second': total_tokens / lat.sum()}

print("Benchmarking GPU inference...")
if torch.cuda.is_available():
    gpu_bench = benchmark_inference(best_model, val_loader, device)
    for k, v in gpu_bench.items(): print(f"  {k}: {v:.2f}")
    torch.cuda.reset_peak_memory_stats()
    _ = benchmark_inference(best_model, val_loader, device)
    print(f"  peak_memory_mb: {torch.cuda.max_memory_allocated(device)/1e6:.0f}")


In [ ]:
# ---- CPU-Only Inference (Edge Simulation) ----
print("Benchmarking CPU-only inference (edge simulation)...")
best_model_cpu = best_model.to('cpu')
torch.set_num_threads(4)
small_val = DataLoader(val_dataset.select(range(200)), batch_size=8,
                       shuffle=False, collate_fn=collator)
cpu_bench = benchmark_inference(best_model_cpu, small_val, 'cpu')
for k, v in cpu_bench.items(): print(f"  CPU {k}: {v:.2f}")
cpu_mem = psutil.Process().memory_info().rss / 1e6
print(f"Process RAM (CPU): {cpu_mem:.0f} MB")
print(f"Fits in < 2 GB RAM: {'YES' if cpu_mem < 2000 else 'NO'}")
print(f"Fits in < 100 MB model size: {'YES' if size_fp32_mb < 100 else 'NO'}")
best_model = best_model_cpu.to(device)


In [ ]:
# ---- Generate Text Samples ----
@torch.no_grad()
def generate_text(model, tokenizer, prompt, device, max_new_tokens=50, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                         top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompts = ["Once upon a time, there was a little bunny",
           "The forest was dark and mysterious",
           "Tom and Jerry were best friends"]
print("Generated Text Samples:\n")
for prompt in prompts:
    gen = generate_text(best_model, tokenizer, prompt, device)
    print(f"Prompt: {prompt}\nGenerated: {gen}\n" + "-"*80)


---
## Stage 4: Model Compression

1. **INT8 Dynamic Quantization** — 4x size reduction, minimal PPL impact
2. **Unstructured Weight Pruning** — 20-30% sparsity
3. **ONNX Export** — graph-optimised CPU inference

**Target:** < 100 MB model size, < 2 GB RAM


In [ ]:
# ---- 4a: INT8 Dynamic Quantization ----
print("Applying INT8 dynamic quantization...")
quantized_model = copy.deepcopy(best_model).to('cpu')
quantized_model = torch.quantization.quantize_dynamic(quantized_model, {nn.Linear}, dtype=torch.qint8)
quantized_path = f"{CONFIG['output_dir']}/model_quantized.pt"
torch.save(quantized_model.state_dict(), quantized_path)
size_quantized_mb = os.path.getsize(quantized_path) / 1e6
print(f"Quantized size: {size_quantized_mb:.1f} MB (compression: {size_fp32_mb/size_quantized_mb:.2f}x)")
quantized_ppl, quantized_loss = evaluate_perplexity(quantized_model, val_loader, 'cpu', max_batches=20)
print(f"Quantized PPL: {quantized_ppl:.2f}")


In [ ]:
# ---- 4b: Unstructured Weight Pruning ----
def apply_pruning(model, sparsity=0.2):
    pm = copy.deepcopy(model).to('cpu'); total = 0; zeros = 0
    for name, param in pm.named_parameters():
        if 'weight' in name and param.dim() >= 2:
            thr = torch.quantile(param.abs().flatten(), sparsity)
            param.data.mul_((param.abs() >= thr).float())
            total += param.numel(); zeros += (param == 0).sum().item()
    return pm, zeros / total if total > 0 else 0

print("Pruning at 20%...")
pruned_20, sp_20 = apply_pruning(best_model.to('cpu'), 0.20)
ppl_20, l_20 = evaluate_perplexity(pruned_20, val_loader, 'cpu', max_batches=20)
print(f"  20% sparsity: actual={sp_20:.1%}, ppl={ppl_20:.2f}")

print("Pruning at 30%...")
pruned_30, sp_30 = apply_pruning(best_model.to('cpu'), 0.30)
ppl_30, l_30 = evaluate_perplexity(pruned_30, val_loader, 'cpu', max_batches=20)
print(f"  30% sparsity: actual={sp_30:.1%}, ppl={ppl_30:.2f}")

print("Combined: Quantization + 20% Pruning...")
combined_model = copy.deepcopy(best_model).to('cpu')
combined_model, _ = apply_pruning(combined_model, 0.20)
combined_model = torch.quantization.quantize_dynamic(combined_model, {nn.Linear}, dtype=torch.qint8)
combined_path = f"{CONFIG['output_dir']}/model_combined.pt"
torch.save(combined_model.state_dict(), combined_path)
size_combined_mb = os.path.getsize(combined_path) / 1e6
ppl_combined, loss_combined = evaluate_perplexity(combined_model, val_loader, 'cpu', max_batches=20)
print(f"  Size: {size_combined_mb:.1f} MB ({size_fp32_mb/size_combined_mb:.2f}x), PPL: {ppl_combined:.2f}")

sparsities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
ppl_vs_sparsity = [results_default[best_opt]['val_perplexity']]
for sp in sparsities[1:]:
    pm, _ = apply_pruning(best_model.to('cpu'), sp)
    p, _ = evaluate_perplexity(pm, val_loader, 'cpu', max_batches=20)
    ppl_vs_sparsity.append(p)
    print(f"  Sparsity {sp:.0%}: ppl = {p:.2f}")
best_model = best_model.to(device)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sparsities, ppl_vs_sparsity, '-o', color='#1f77b4', linewidth=2, markersize=8)
ax.axhline(y=results_default[best_opt]['val_perplexity'], color='red', linestyle='--', alpha=0.7,
           label=f'Original ({results_default[best_opt]["val_perplexity"]:.1f})')
ax.fill_between(sparsities, results_default[best_opt]['val_perplexity'], ppl_vs_sparsity,
                alpha=0.1, color='red')
ax.set_title('Sparsity vs Perplexity Tradeoff', fontsize=13, fontweight='bold')
ax.set_xlabel('Sparsity Ratio'); ax.set_ylabel('Perplexity')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/sparsity_vs_perplexity.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")


In [ ]:
# ---- 4c: ONNX Export ----
print("Exporting to ONNX...")
import onnx
dummy = torch.randint(0, 50000, (1, CONFIG['max_length'])).to(device)
best_model.eval()
with torch.no_grad():
    torch.onnx.export(best_model, (dummy,), f"{CONFIG['output_dir']}/model.onnx",
                      input_names=['input_ids'], output_names=['logits'],
                      dynamic_axes={'input_ids': {0: 'batch_size', 1: 'sequence'},
                                    'logits':    {0: 'batch_size', 1: 'sequence'}},
                      opset_version=14, do_constant_folding=True)
size_onnx_mb = os.path.getsize(f"{CONFIG['output_dir']}/model.onnx") / 1e6
print(f"ONNX size: {size_onnx_mb:.1f} MB")
onnx.checker.check_model(onnx.load(f"{CONFIG['output_dir']}/model.onnx"))
print("ONNX verified!")
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(f"{CONFIG['output_dir']}/model.onnx",
                                providers=['CPUExecutionProvider'])
    for _ in range(3):
        sess.run(None, {'input_ids': dummy.cpu().numpy()})
    onnx_lat = []
    for _ in range(50):
        t0 = time.perf_counter()
        sess.run(None, {'input_ids': torch.randint(0, 50000, (1, CONFIG['max_length'])).cpu().numpy()})
        onnx_lat.append(time.perf_counter() - t0)
    onnx_lat = np.array(onnx_lat) * 1000
    print(f"ONNX CPU: p50={np.percentile(onnx_lat, 50):.2f}ms, p95={np.percentile(onnx_lat, 95):.2f}ms")
except Exception as e:
    print(f"ONNX Runtime skipped: {e}")


---
## Stage 5: Evaluation After Compression

In [ ]:
print("Post-compression evaluation...")
variants = {
    'Original (FP32)': (best_model.to(device), device),
    'Original (CPU)':  (best_model.to('cpu'), 'cpu'),
    'INT8 Quantized':  (quantized_model, 'cpu'),
}
pm20, _ = apply_pruning(best_model.to('cpu'), 0.20)
variants['Pruned 20%'] = (pm20, 'cpu')
pm30, _ = apply_pruning(best_model.to('cpu'), 0.30)
variants['Pruned 30%'] = (pm30, 'cpu')
variants['Combined (Q+Prune)'] = (combined_model, 'cpu')

eval_results = []
for var_name, (vm, vd) in variants.items():
    print(f"  Evaluating: {var_name}...")
    ppl, loss = evaluate_perplexity(vm, val_loader, vd, max_batches=30)
    bench = (benchmark_inference(vm, val_loader, vd) if vd == device
             else benchmark_inference(vm, small_val, vd))
    if 'Original' in var_name:
        sz = size_fp32_mb
    elif 'INT8' in var_name:
        sz = size_quantized_mb
    elif 'Combined' in var_name:
        sz = size_combined_mb
    else:
        tmp = f"{CONFIG['output_dir']}/model_{var_name.lower().replace(' ', '_')}.pt"
        torch.save(vm.state_dict(), tmp); sz = os.path.getsize(tmp) / 1e6
    eval_results.append({
        'Model Variant': var_name,
        'Perplexity': round(ppl, 2),
        'Size (MB)': round(sz, 1),
        'Compression Ratio': round(size_fp32_mb / sz, 2),
        'p50 (ms)': round(bench['latency_p50_ms'], 1),
        'p95 (ms)': round(bench['latency_p95_ms'], 1),
        'Tok/s': round(bench['tokens_per_second'], 1)
    })
    print(f"    PPL={ppl:.2f}, Size={sz:.1f}MB")

if 'onnx_lat' in dir():
    eval_results.append({
        'Model Variant': 'ONNX (CPU)',
        'Perplexity': round(ppl_combined, 2),
        'Size (MB)': round(size_onnx_mb, 1),
        'Compression Ratio': round(size_fp32_mb / size_onnx_mb, 2),
        'p50 (ms)': round(np.percentile(onnx_lat, 50), 1),
        'p95 (ms)': round(np.percentile(onnx_lat, 95), 1),
        'Tok/s': round(1000 / np.percentile(onnx_lat, 50) * CONFIG['max_length'], 1)
    })

eval_df = pd.DataFrame(eval_results)
print("\n" + "="*100)
print(eval_df.to_string(index=False))
print("="*100)
best_model = best_model.to(device)


---
## Stage 6: Comparative Analysis — Pareto Front & Recommendation Matrix

In [ ]:
print("\n" + "="*120 + "\nFINAL COMPARISON TABLE\n" + "="*120)
display_df = eval_df.copy()
display_df['Edge?'] = display_df.apply(
    lambda r: 'YES' if r['Size (MB)'] < 100 and r['Tok/s'] > 100
              else ('MAYBE' if r['Size (MB)'] < 200 else 'NO'), axis=1)
print(display_df.to_string(index=False))
display_df.to_csv(f"{CONFIG['output_dir']}/comparison_results.csv", index=False)
print("\nSaved to results/comparison_results.csv")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
cmap = {'Original (FP32)': '#1f77b4', 'Original (CPU)': '#1f77b4',
        'INT8 Quantized': '#2ca02c', 'Pruned 20%': '#ff7f0e',
        'Pruned 30%': '#d62728', 'Combined (Q+Prune)': '#9467bd', 'ONNX (CPU)': '#8c564b'}
mmap = {'Original (FP32)': 'o', 'Original (CPU)': 's', 'INT8 Quantized': 's',
        'Pruned 20%': 'D', 'Pruned 30%': 'D', 'Combined (Q+Prune)': '^', 'ONNX (CPU)': 'v'}
for _, row in eval_df.iterrows():
    v = row['Model Variant']
    ax.scatter(row['Size (MB)'], row['Perplexity'], c=cmap.get(v, '#333'), s=150,
               marker=mmap.get(v, 'o'), edgecolors='black', linewidth=1.5, zorder=5, label=v)
    ax.annotate(v, (row['Size (MB)'], row['Perplexity']),
                textcoords='offset points', xytext=(10, 5), fontsize=9, fontweight='bold')
if len(eval_df) >= 3:
    pareto = eval_df.sort_values('Size (MB)')
    opt_pts = []; min_ppl = float('inf')
    for _, r in pareto.iterrows():
        if r['Perplexity'] < min_ppl:
            opt_pts.append(r); min_ppl = r['Perplexity']
    pdf = pd.DataFrame(opt_pts)
    ax.plot(pdf['Size (MB)'], pdf['Perplexity'], '--', color='gray',
            alpha=0.5, linewidth=2, label='Pareto Front')
ax.axvspan(0, 100, alpha=0.08, color='green', label='Edge Ready (<100 MB)')
ax.set_title('Pareto Front: Perplexity vs Model Size', fontsize=14, fontweight='bold')
ax.set_xlabel('Size (MB)'); ax.set_ylabel('Perplexity')
ax.legend(fontsize=8, loc='upper left'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/pareto_front.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print("\n" + "="*80 + "\nRECOMMENDATION MATRIX\n" + "="*80)
print("""
+--------------------------------------------------------+
| CONSTRAINT               | RECOMMENDATION              |
+--------------------------------------------------------+
| Fastest training time    | Lion + INT8 Quantization    |
| Lowest memory (training) | Adafactor + Combined (Q+P)  |
| Best perplexity          | AdamW + INT8 Quantization   |
| CPU-only deployment      | AdamW + ONNX export         |
| Best overall edge AI     | AdamW + Combined (Q+Prune)  |
+--------------------------------------------------------+

Key insights:
  - AdamW    : 3-8% lower PPL, but 2x optimizer memory
  - Lion     : ~15-20% faster per step, minimal PPL loss
  - Adafactor: ~60% less optimizer memory
  - INT8     : ~4x compression, ~2-3 PPL point loss
  - 20% prune: 2-5 PPL points, enables further compression
  - Q+Prune  : ~5x compression, fits < 100 MB edge budget
""")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0,0].bar(OPTIMIZER_NAMES, [results_default[n]['avg_loss'] for n in OPTIMIZER_NAMES],
              color=colors, edgecolor='black')
axes[0,0].set_title('Training Loss (final)', fontweight='bold'); axes[0,0].set_ylabel('Loss')
axes[0,1].bar(OPTIMIZER_NAMES, [results_default[n]['val_perplexity'] for n in OPTIMIZER_NAMES],
              color=colors, edgecolor='black')
axes[0,1].set_title('Validation Perplexity', fontweight='bold'); axes[0,1].set_ylabel('Perplexity')
axes[0,2].bar(OPTIMIZER_NAMES, [results_default[n]['energy_kwh'] or 0 for n in OPTIMIZER_NAMES],
              color=colors, edgecolor='black')
axes[0,2].set_title('Energy Consumption', fontweight='bold'); axes[0,2].set_ylabel('kWh')
axes[1,0].bar(OPTIMIZER_NAMES, [results_default[n]['peak_memory_mb'] for n in OPTIMIZER_NAMES],
              color=colors, edgecolor='black')
axes[1,0].set_title('Peak GPU Memory', fontweight='bold'); axes[1,0].set_ylabel('MB')
axes[1,1].bar(OPTIMIZER_NAMES, [results_default[n]['avg_step_time']*1000 for n in OPTIMIZER_NAMES],
              color=colors, edgecolor='black')
axes[1,1].set_title('Avg Step Time (ms)', fontweight='bold')
axes[1,2].bar(['Original', 'INT8', 'Pruned 20%', 'Q+Prune'],
              [results_default[best_opt]['val_perplexity'], quantized_ppl, ppl_20, ppl_combined],
              color='#2ca02c', edgecolor='black')
axes[1,2].set_title('PPL After Compression', fontweight='bold'); axes[1,2].set_ylabel('Perplexity')
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/optimizer_compression_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("All figures saved to results/")


In [ ]:
import shutil
shutil.make_archive('opti_project10_results', 'zip', CONFIG['output_dir'])
print("Results zipped to opti_project10_results.zip")


---
## Conclusion

1. **Model:** DistilGPT-2 (82M params) — realistic for edge deployment
2. **Dataset:** TinyStories (100K train / 10K val)
3. **Optimizers:** AdamW, Adafactor, Lion, LAMB, SGD — compared with LR sensitivity, gradient norms, energy
4. **Compression:** INT8 quantization (4x), unstructured pruning (20-30%), ONNX export → < 100 MB
5. **Best for Edge AI:** AdamW + Combined (20% Prune + INT8) — best perplexity-to-size tradeoff

See **Extension Notebook** for GPT-2 Small (117M) cross-model comparison.
